# Colab Quickstart: Qwen2.5-3B

This notebook sets up the project on Google Colab Free, downloads Qwen2.5-3B, and runs a quick benchmark command.

## 1) Clone Repo and Install Dependencies

In [ ]:
# Replace with your repository URL
REPO_URL = "<your-repo-url>"

!git clone {REPO_URL} CSE-495B_NLP-main
%cd CSE-495B_NLP-main
!pip install -U pip
!pip install -r requirements.txt

## 2) Optional Hugging Face Login

Recommended for better rate limits.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3) Configure Cache Paths

In [ ]:
import os

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache"
os.environ["TORCH_HOME"] = "/content/hf_cache"

print("HF_HOME=", os.environ["HF_HOME"])

## 4) Download Qwen2.5-3B

In [ ]:
!python download_models.py --models qwen2.5-3b --cache-dir /content/hf_cache

## 4.1) Fix Missing Modules (only if needed)

If your cloned repo does not yet include `src/models` and `src/data`, run the next cell once before experiment cells.

In [ ]:
from pathlib import Path
from textwrap import dedent

# Force-create missing modules for repos that do not yet include src/models or src/data
models_dir = Path("src/models")
data_dir = Path("src/data")
models_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

(models_dir / "__init__.py").write_text(
    "from .loader import ModelLoader\n\n__all__ = ['ModelLoader']\n",
    encoding="utf-8",
)

(models_dir / "loader.py").write_text(
    dedent(
        """
        from typing import Optional, Tuple, Any
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        SUPPORTED = {
            "qwen2.5-3b": "Qwen/Qwen2.5-3B-Instruct",
            "mistral-7b": "mistralai/Mistral-7B-Instruct-v0.2",
            "llama2-7b": "meta-llama/Llama-2-7b-chat-hf",
            "llama2-13b": "meta-llama/Llama-2-13b-chat-hf",
            "mixtral-8x7b": "mistralai/Mixtral-8x7B-Instruct-v0.1",
        }

        class ModelLoader:
            def __init__(self, model_name: str, quantization: Optional[str] = "4bit", cache_dir: Optional[str] = None):
                if model_name not in SUPPORTED:
                    raise ValueError(f"Unsupported model: {model_name}")
                self.model_id = SUPPORTED[model_name]
                self.quantization = quantization
                self.cache_dir = cache_dir

            def load(self) -> Tuple[Any, Any]:
                tokenizer = AutoTokenizer.from_pretrained(self.model_id, cache_dir=self.cache_dir, trust_remote_code=True)
                if tokenizer.pad_token is None:
                    tokenizer.pad_token = tokenizer.eos_token

                kwargs = {"cache_dir": self.cache_dir, "trust_remote_code": True, "low_cpu_mem_usage": True}
                if torch.cuda.is_available() and str(self.quantization).lower() in {"4bit", "8bit"}:
                    if str(self.quantization).lower() == "4bit":
                        kwargs["quantization_config"] = BitsAndBytesConfig(
                            load_in_4bit=True,
                            bnb_4bit_quant_type="nf4",
                            bnb_4bit_compute_dtype=torch.float16,
                            bnb_4bit_use_double_quant=True,
                        )
                    else:
                        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
                    kwargs["device_map"] = "auto"
                else:
                    kwargs["torch_dtype"] = torch.float16 if torch.cuda.is_available() else torch.float32
                    kwargs["device_map"] = "auto" if torch.cuda.is_available() else "cpu"

                model = AutoModelForCausalLM.from_pretrained(self.model_id, **kwargs)
                model.eval()
                return model, tokenizer
        """
    ).strip() + "\n",
    encoding="utf-8",
)

(data_dir / "__init__.py").write_text(
    "from .datasets import get_dataset, ReasoningSample\n\n__all__ = ['get_dataset', 'ReasoningSample']\n",
    encoding="utf-8",
)

(data_dir / "datasets.py").write_text(
    dedent(
        """
        from dataclasses import dataclass
        from typing import Optional, List, Dict
        from datasets import load_dataset

        @dataclass
        class ReasoningSample:
            idx: int
            question: str
            answer: str
            context: Optional[str] = None

        class _Base:
            task_type = "general"
            def __init__(self, split="test", max_samples=None, seed=42):
                self.split = split
                self.max_samples = max_samples
                self.seed = seed
                self.samples: List[ReasoningSample] = []

            def __iter__(self):
                return iter(self.samples)

            def __len__(self):
                return len(self.samples)

            def get_references(self) -> List[Dict[str, Optional[str]]]:
                return [{"answer": s.answer, "question": s.question, "context": s.context} for s in self.samples]

            def _cap(self):
                if self.max_samples is not None:
                    self.samples = self.samples[:self.max_samples]

        class GSM8K(_Base):
            task_type = "arithmetic"
            def load(self):
                ds = load_dataset("gsm8k", "main", split=self.split)
                self.samples = [ReasoningSample(i, r["question"], r["answer"], None) for i, r in enumerate(ds)]
                self._cap()

        class SQuAD(_Base):
            task_type = "reading"
            def load(self):
                split = self.split if self.split in {"train", "validation"} else "validation"
                ds = load_dataset("squad", split=split)
                out = []
                for i, r in enumerate(ds):
                    ans = r.get("answers", {}).get("text", [])
                    out.append(ReasoningSample(i, r.get("question", ""), ans[0] if ans else "", r.get("context", "")))
                self.samples = out
                self._cap()

        class Fallback(_Base):
            def __init__(self, name, **kwargs):
                super().__init__(**kwargs)
                self.name = name

            def load(self):
                self.samples = [
                    ReasoningSample(0, "If Alice has 2 apples and gets 3 more, how many?", "5", None),
                    ReasoningSample(1, "Which is larger: 9 or 12?", "12", None),
                ]
                self._cap()

        def get_dataset(dataset_name: str, split="test", max_samples=None, seed=42):
            name = dataset_name.lower()
            if name == "gsm8k":
                return GSM8K(split=split, max_samples=max_samples, seed=seed)
            if name in {"squad", "squad_v1", "reading"}:
                return SQuAD(split=split, max_samples=max_samples, seed=seed)
            return Fallback(name, split=split, max_samples=max_samples, seed=seed)
        """
    ).strip() + "\n",
    encoding="utf-8",
)

# Verify imports immediately
from src.models import ModelLoader
from src.data import get_dataset
print("Bootstrap complete: src.models and src.data imports OK")

## 5) Quick Sanity Run (10 samples)

In [ ]:
!python -m experiments.run_experiment \
  --model qwen2.5-3b \
  --prompting cot \
  --decoding greedy \
  --dataset gsm8k \
  --max-samples 10 \
  --output-dir /content/results_quick

## 6) Colab Preset Benchmark

In [ ]:
!python -m experiments.run_experiment --config experiments/configs/colab_free.yaml

## 7) Optional: Save Results to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import shutil

drive.mount('/content/drive')

src_dir = Path('/content/results_quick')
drive_root = Path('/content/drive/MyDrive/CSE495B_results_quick')
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
dst_dir = drive_root / f'results_snapshot_{timestamp}'

if not src_dir.exists():
    print(f'Source not found: {src_dir}')
    print('Run the experiment cells first.')
else:
    dst_dir.mkdir(parents=True, exist_ok=True)
    copied = 0
    for src in src_dir.rglob('*'):
        if src.is_file():
            rel = src.relative_to(src_dir)
            dst = dst_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            copied += 1

    print('Saved results snapshot to Drive.')
    print(f'Destination: {dst_dir}')
    print(f'Files copied: {copied}')